<a href="https://colab.research.google.com/github/ebtihalallien-alt/Genspark-Agentic-New-Curation/blob/main/Stage_6_Bounded_Agent_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Genspark Agentic News Curation System
Stage 6: Bounded Agent Loop

Flow:
OBSERVE -> DECIDE -> VALIDATE -> ACT -> OBSERVE RESULT

Project:
Genspark Agentic News Curation System Inspired by Genspark
"""

# Approved news tools from the project proposal

ALLOWED_TOOLS = {
    "newsdata",
    "gnews",
    "google_news_rss"
}


MAX_TURNS = 3


def observe(user_query):

    return {
        "user_query": user_query
    }


def decide(state):

    user_query = state["user_query"]

    if user_query.strip() == "":
        return {
            "action": "ask_user",
            "tool": None,
            "reason": "A news topic is required."
        }

    return {
        "action": "tool",
        "tool": "newsdata",
        "reason": "Current news requires an approved news source."
    }


def validate(action):

    allowed_actions = {
        "tool",
        "ask_user",
        "answer",
        "escalate"
    }

    if action["action"] not in allowed_actions:

        return False, "Action is not allowed."

    if action["action"] == "tool":

        if action["tool"] not in ALLOWED_TOOLS:

            return False, "Requested tool is not approved."

    return True, "Action is valid."


def act(action, state):

    if action["action"] == "tool":

        return {
            "status": "TOOL_REQUEST_APPROVED",
            "tool": action["tool"],
            "message": "The approved news tool is ready to collect public news."
        }

    if action["action"] == "ask_user":

        return {
            "status": "WAITING_FOR_USER",
            "message": "More information is required."
        }

    if action["action"] == "answer":

        return {
            "status": "COMPLETED",
            "message": "The agent can provide the prepared news result."
        }

    return {
        "status": "ESCALATED",
        "message": "The request requires escalation."
    }


def run_agent(user_query):

    state = observe(user_query)

    print("Agent Run Started")
    print("-----------------")

    for turn in range(1, MAX_TURNS + 1):

        print()
        print("Turn:", turn)

        action = decide(state)

        print("Decision:", action)

        is_valid, validation_message = validate(action)

        print("Validation:", validation_message)

        if not is_valid:

            print("Terminal State: FAILED_SAFELY")

            return

        result = act(action, state)

        print("Action Result:", result)

        state["last_result"] = result

        if result["status"] in {
            "COMPLETED",
            "WAITING_FOR_USER",
            "ESCALATED",
            "FAILED_SAFELY"
        }:

            print()
            print("Terminal State:", result["status"])

            return

        if result["status"] == "TOOL_REQUEST_APPROVED":

            print()
            print("Terminal State: READY_FOR_TOOL_EXECUTION")

            return

    print()
    print("Terminal State: BUDGET_EXHAUSTED")


# Test the agent

user_query = "Find recent news about Artificial Intelligence."

run_agent(user_query)